# **1. Thermal_Human_Detection**

In [ ]:

import cv2
import time
import platform
from datetime import datetime
from ultralytics import YOLO

CAMERA_SOURCE = 0

CONFIDENCE = 0.40
ALERT_COOLDOWN = 5

NIGHT_START = "22:00"
NIGHT_END = "06:00"

# True = Testing anytime
# False = Night-time detection only
ACTIVE = True

# LOAD YOLO MODEL
model = YOLO("yolov8n.pt")

print("YOLO model loaded successfully!")


# CAMERA INITIALIZATION
cap = cv2.VideoCapture(CAMERA_SOURCE, cv2.CAP_DSHOW)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    print("ERROR: Camera not detected!")
    exit()

print("Camera connected successfully!")

# NIGHT-TIME CHECK
def is_night_time():

    if ACTIVE:
        return True

    current = datetime.now().strftime("%H:%M")

    if NIGHT_START > NIGHT_END:
        return current >= NIGHT_START or current < NIGHT_END

    return NIGHT_START <= current < NIGHT_END


# THERMAL COLOR EFFECT
def apply_thermal_effect(frame):

    # Convert normal camera image to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Improve image contrast
    gray = cv2.equalizeHist(gray)

    # Apply thermal-style color mapping
    thermal = cv2.applyColorMap(
        gray,
        cv2.COLORMAP_JET
    )

    return thermal


# SECURITY ALERT
last_alert_time = 0

def security_alert():

    global last_alert_time

    current_time = time.time()

    if current_time - last_alert_time >= ALERT_COOLDOWN:

        print("SECURITY ALERT!")
        print("HUMAN DETECTED!")
        print("Time:", datetime.now())
        if platform.system() == "Windows":

            import winsound

            winsound.Beep(1500, 500)

        last_alert_time = current_time


# HUMAN DETECTION
def detect_humans(original_frame, thermal_frame):

    # Detect using original image for better accuracy
    results = model.predict(
        original_frame,
        conf=CONFIDENCE,
        classes=[0],
        verbose=False
    )

    human_count = 0

    for result in results:

        for box in result.boxes:

            human_count += 1

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].tolist()
            )

            confidence = float(box.conf[0])

            # Red bounding box
            cv2.rectangle(
                thermal_frame,
                (x1, y1),
                (x2, y2),
                (0, 0, 255),
                2
            )

            label = f"HUMAN {confidence:.2f}"

            cv2.putText(
                thermal_frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

    return thermal_frame, human_count


# 8. MAIN LOOP
print("Thermal Human Detection Started!")

while True:

    ret, frame = cap.read()

    if not ret:
        print("Camera frame not received!")
        break

    # Mirror camera
    frame = cv2.flip(frame, 1)

    # Apply simulated thermal view
    thermal_frame = apply_thermal_effect(frame)

    active = is_night_time()

    human_count = 0

    if active:

        thermal_frame, human_count = detect_humans(
            frame,
            thermal_frame
        )

        if human_count > 0:

            status = "INTRUDER DETECTED!"
            status_color = (0, 0, 255)

            security_alert()

        else:

            status = "AREA SECURE"
            status_color = (0, 255, 0)

    else:

        status = "MONITORING INACTIVE"
        status_color = (255, 255, 255)

    # ======================================
    # 9. DISPLAY INFORMATION
    # ======================================

    height, width = thermal_frame.shape[:2]

    # Header background
    cv2.rectangle(
        thermal_frame,
        (0, 0),
        (width, 110),
        (20, 20, 20),
        -1
    )

    # System title
    cv2.putText(
        thermal_frame,
        "THERMAL HUMAN DETECTION SYSTEM",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # Security status
    cv2.putText(
        thermal_frame,
        status,
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        status_color,
        2
    )

    # Human count
    cv2.putText(
        thermal_frame,
        f"Humans Detected: {human_count}",
        (20, 95),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # Timestamp
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    cv2.putText(
        thermal_frame,
        timestamp,
        (20, height - 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # Show simulated thermal output
    cv2.imshow(
        "Thermal Human Detection",
        thermal_frame
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break



cap.release()

cv2.destroyAllWindows()

print("Thermal Detection System Stopped!")


# **2. Human Detection**

In [ ]:
import cv2
import time
import platform
from datetime import datetime
from ultralytics import YOLO


SOURCE = 0  #For laptop camera


# SOURCE = ".................."     # thermal camera:


# SOURCE = r"C:\Users\DELL\Downloads\thermal_image.png"     #thermal image

# Night-time security schedule
NIGHT_START = "22:00"
NIGHT_END = "06:00"

# Set True for testing at any time
_ACTIVE = True

# Detection settings
CONFIDENCE = 0.40
ALERT_COOLDOWN = 5

# LOAD YOLO MODEL
model = YOLO("yolov8n.pt")
print("YOLO model loaded successfully!")



# NIGHT-TIME DETECTION
def is_night_time():

    if _ACTIVE:
        return True

    current_time = datetime.now().strftime("%H:%M")

    if NIGHT_START > NIGHT_END:
        return (
            current_time >= NIGHT_START
            or current_time < NIGHT_END
        )

    return NIGHT_START <= current_time < NIGHT_END


# SECURITY ALERT
last_alert_time = 0

def generate_alert():

    global last_alert_time

    current_time = time.time()

    if current_time - last_alert_time >= ALERT_COOLDOWN:

        print("SECURITY ALERT!")
        print("UNAUTHORIZED HUMAN DETECTED")
        print("Time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

        # Windows alarm sound
        if platform.system() == "Windows":

            import winsound

            winsound.Beep(1500, 1000)

        else:

            print("\a")

        last_alert_time = current_time


# HUMAN DETECTION USING YOLO

def detect_humans(frame):

    results = model.predict(
        source=frame,
        conf=CONFIDENCE,
        classes=[0],  # COCO class 0 = Person
        verbose=False
    )

    human_detected = False

    for result in results:

        for box in result.boxes:

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].tolist()
            )

            confidence = float(box.conf[0])

            human_detected = True

            # Draw bounding box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 0, 255),
                2
            )

            # Display detection information
            label = f"Human: {confidence:.2f}"

            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

    return frame, human_detected


# CAMERA / VIDEO / IMAGE INPUT
print("Starting Thermal Human Detection System...")

# Check whether the source is an image
if isinstance(SOURCE, str) and SOURCE.lower().endswith(
    (".jpg", ".jpeg", ".png", ".bmp")
):

    frame = cv2.imread(SOURCE)

    if frame is None:
        raise ValueError("Unable to load image!")

    image_mode = True

else:

    cap = cv2.VideoCapture(SOURCE)

    if not cap.isOpened():
        raise ValueError("Unable to connect to camera/video!")

    image_mode = False


# MAIN SECURITY MONITORING LOOP
while True:

    if image_mode:

        frame = cv2.imread(SOURCE)

    else:

        ret, frame = cap.read()

        if not ret:
            print("Unable to read camera frame.")
            break

    # Check security schedule
    active = is_night_time()

    if active:

        # Detect humans
        frame, human_detected = detect_humans(frame)

        if human_detected:

            generate_alert()

            status = "INTRUDER DETECTED!"

            color = (0, 0, 255)

        else:

            status = "AREA SECURE - NO HUMAN"

            color = (0, 255, 0)

    else:

        status = "SECURITY MONITORING INACTIVE"

        color = (255, 255, 0)

    # Display system status
    cv2.putText(
        frame,
        status,
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        color,
        2
    )

    # Display current time
    current_time = datetime.now().strftime("%H:%M:%S")

    cv2.putText(
        frame,
        current_time,
        (20, 70),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow(
        "Thermal Human Detection",
        frame
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

    # Static image: process only once
    if image_mode:
        cv2.waitKey(0)
        break


# RELEASE RESOURCES
if not image_mode:
    cap.release()

cv2.destroyAllWindows()

print("Thermal Security System Stopped.")

YOLO model loaded successfully!
Starting Thermal Human Detection System...

SECURITY ALERT!
UNAUTHORIZED HUMAN DETECTED
Time: 2026-11-04 21:08:24


SECURITY ALERT!
UNAUTHORIZED HUMAN DETECTED
Time: 2026-11-04 21:08:29


SECURITY ALERT!
UNAUTHORIZED HUMAN DETECTED
Time: 2026-11-04 21:08:34


SECURITY ALERT!
UNAUTHORIZED HUMAN DETECTED
Time: 2026-11-04 21:08:39



KeyboardInterrupt: 

: 

# **3. Line-Crossing Based People Counter Using Computer Vision**

In [ ]:
import cv2
from ultralytics import YOLO

# ==========================================
# 1. CONFIGURATION
# ==========================================

# Custom video path
VIDEO_PATH = "entrance_video.mp4"

# YOLO model
MODEL_PATH = "yolov8n.pt"

CONFIDENCE = 0.35

# Virtual counting line position
# 0.50 means 50% of video height
LINE_POSITION = 0.50

# Direction configuration:
# Top to Bottom = IN
# Bottom to Top = OUT

# ==========================================
# 2. LOAD YOLO MODEL
# ==========================================

print("Loading YOLO model...")

model = YOLO(MODEL_PATH)

print("YOLO model loaded successfully!")

# ==========================================
# 3. OPEN CUSTOM VIDEO
# ==========================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise ValueError(
        "Unable to open video. Check VIDEO_PATH!"
    )

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video loaded successfully!")
print("Resolution:", width, "x", height)

# Virtual horizontal line
LINE_Y = int(height * LINE_POSITION)

# ==========================================
# 4. PEOPLE COUNTING VARIABLES
# ==========================================

in_count = 0
out_count = 0

# Store previous center position of each person
previous_positions = {}

# Store unique IDs already counted
counted_in = set()
counted_out = set()

# ==========================================
# 5. MAIN VIDEO PROCESSING LOOP
# ==========================================

while True:

    ret, frame = cap.read()

    if not ret:
        print("Video processing completed!")
        break

    # ======================================
    # 6. YOLO + BYTETRACK
    # ======================================

    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0],  # Person only
        conf=CONFIDENCE,
        verbose=False
    )

    # Draw virtual counting line
    cv2.line(
        frame,
        (0, LINE_Y),
        (width, LINE_Y),
        (0, 255, 255),
        3
    )

    cv2.putText(
        frame,
        "COUNTING LINE",
        (20, LINE_Y - 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )

    # ======================================
    # 7. PROCESS TRACKED PEOPLE
    # ======================================

    for result in results:

        if result.boxes is None:
            continue

        if result.boxes.id is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()

        track_ids = result.boxes.id.int().cpu().tolist()

        for box, track_id in zip(boxes, track_ids):

            x1, y1, x2, y2 = map(int, box)

            # Calculate person's center point
            center_x = int((x1 + x2) / 2)
            center_y = int((y1 + y2) / 2)

            # ==================================
            # 8. LINE CROSSING DETECTION
            # ==================================

            if track_id in previous_positions:

                previous_y = previous_positions[track_id]

                # TOP TO BOTTOM = IN
                if (
                    previous_y < LINE_Y
                    and center_y >= LINE_Y
                    and track_id not in counted_in
                ):

                    in_count += 1
                    counted_in.add(track_id)

                    print(
                        f"PERSON {track_id} ENTERED | IN: {in_count}"
                    )

                # BOTTOM TO TOP = OUT
                elif (
                    previous_y > LINE_Y
                    and center_y <= LINE_Y
                    and track_id not in counted_out
                ):

                    out_count += 1
                    counted_out.add(track_id)

                    print(
                        f"PERSON {track_id} EXITED | OUT: {out_count}"
                    )

            # Update person's previous position
            previous_positions[track_id] = center_y

            # ==================================
            # 9. DRAW PERSON BOUNDING BOX
            # ==================================

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Draw center point
            cv2.circle(
                frame,
                (center_x, center_y),
                5,
                (0, 0, 255),
                -1
            )

            # Display tracking ID
            cv2.putText(
                frame,
                f"Person ID: {track_id}",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # ======================================
    # 10. CALCULATE CURRENT PEOPLE INSIDE
    # ======================================

    current_inside = max(0, in_count - out_count)

    # ======================================
    # 11. DISPLAY COUNTING DASHBOARD
    # ======================================

    cv2.rectangle(
        frame,
        (10, 10),
        (340, 145),
        (30, 30, 30),
        -1
    )

    cv2.putText(
        frame,
        f"IN COUNT: {in_count}",
        (25, 45),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    cv2.putText(
        frame,
        f"OUT COUNT: {out_count}",
        (25, 80),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

    cv2.putText(
        frame,
        f"INSIDE: {current_inside}",
        (25, 115),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    # ======================================
    # 12. DISPLAY VIDEO
    # ======================================

    cv2.imshow(
        "Custom Line Crossing - People Counting",
        frame
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ==========================================
# 13. RELEASE RESOURCES
# ==========================================

cap.release()

cv2.destroyAllWindows()

print("Total IN:", in_count)
print("Total OUT:", out_count)
print("Current Inside:", max(0, in_count - out_count))


Loading YOLO model...
YOLO model loaded successfully!
Video loaded successfully!
Resolution: 1280 x 720
PERSON 3 EXITED | OUT: 1
PERSON 3 ENTERED | IN: 1
PERSON 29 EXITED | OUT: 2
PERSON 1 EXITED | OUT: 3
PERSON 1 ENTERED | IN: 2
PERSON 8 EXITED | OUT: 4
PERSON 114 ENTERED | IN: 3
PERSON 96 EXITED | OUT: 5
PERSON 96 ENTERED | IN: 4
PERSON 131 ENTERED | IN: 5
PERSON 131 EXITED | OUT: 6
PERSON 38 ENTERED | IN: 6
PERSON 38 EXITED | OUT: 7
PERSON 64 ENTERED | IN: 7
PERSON 127 ENTERED | IN: 8
PERSON 128 ENTERED | IN: 9


KeyboardInterrupt: 

# **4. Real-Time Crowd Tracking and Movement Analysis Using YOLOv8**

In [ ]:

import cv2
import time
import random
from collections import defaultdict, deque
from ultralytics import YOLO

# CONFIGURATION
VIDEO_PATH = "entrance_video.mp4"

MODEL_PATH = "yolov8n.pt"

CONFIDENCE = 0.35

TRAIL_LENGTH = 40

# LOAD YOLO MODEL
print("Loading YOLO model...")

model = YOLO(MODEL_PATH)

print("YOLO model loaded successfully!")

# OPEN CUSTOM VIDEO
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise ValueError("Unable to open video!")

print("Video loaded successfully!")

# TRACKING VARIABLES
# Store previous center coordinates for each ID
track_history = defaultdict(
    lambda: deque(maxlen=TRAIL_LENGTH)
)

# Store unique colors for each person
track_colors = {}

# Total unique people detected
unique_people = set()

# GENERATE TRACKING COLORS
def get_track_color(track_id):

    if track_id not in track_colors:

        random.seed(track_id)

        track_colors[track_id] = (
            random.randint(50, 255),
            random.randint(50, 255),
            random.randint(50, 255)
        )

    return track_colors[track_id]


# MAIN TRACKING LOOP
print("Human Tracking Started!")

while True:

    ret, frame = cap.read()

    if not ret:

        print("Video processing completed!")
        break

    # YOLO + BYTETRACK
    results = model.track(
        source=frame,
        persist=True,
        tracker="bytetrack.yaml",
        classes=[0],
        conf=CONFIDENCE,
        verbose=False
    )

    # PROCESS DETECTED HUMANS
    for result in results:

        if result.boxes is None:
            continue

        if result.boxes.id is None:
            continue

        boxes = result.boxes.xyxy.cpu().numpy()

        track_ids = (
            result.boxes.id.int().cpu().tolist()
        )

        confidences = (
            result.boxes.conf.cpu().tolist()
        )

        for box, track_id, confidence in zip(
            boxes, track_ids, confidences
        ):

            x1, y1, x2, y2 = map(int, box)

            # Calculate center position
            center_x = int((x1 + x2) / 2)

            center_y = int((y1 + y2) / 2)

            center = (center_x, center_y)

            # Add tracking ID to unique people
            unique_people.add(track_id)

            # Get person's tracking color
            color = get_track_color(track_id)

            # STORE MOVEMENT COORDINATES
            track_history[track_id].append(center)

            # DRAW BOUNDING BOX
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                color,
                2
            )

            # DISPLAY TRACKING ID
            label = f"ID: {track_id} | {confidence:.2f}"

            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

            # Draw center point
            cv2.circle(
                frame,
                center,
                4,
                color,
                -1
            )

    # DRAW TRAJECTORY / MOVEMENT TRAILS
    for track_id, points in track_history.items():

        color = get_track_color(track_id)

        points = list(points)

        # Draw continuous trajectory
        for i in range(1, len(points)):

            cv2.line(
                frame,
                points[i - 1],
                points[i],
                color,
                3
            )

    # DISPLAY TRACKING DASHBOARD
    height, width = frame.shape[:2]

    cv2.rectangle(
        frame,
        (10, 10),
        (360, 100),
        (30, 30, 30),
        -1
    )

    cv2.putText(
        frame,
        "HUMAN TRACKING SYSTEM",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Unique People: {len(unique_people)}",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    # DISPLAY VIDEO
    cv2.imshow(
        "YOLOv8 - Human Tracking & Trajectory",
        frame
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# RELEASE RESOURCES
cap.release()

cv2.destroyAllWindows()
print("Unique People Detected:", len(unique_people))

print("Person IDs:", sorted(unique_people))




Loading YOLO model...
YOLO model loaded successfully!
Video loaded successfully!
Human Tracking Started!


KeyboardInterrupt: 

: 

# **5. Real-Time Human Activity Heatmap and Crowd Density Analysis System Using YOLOv8 and OpenCV**

In [1]:

import cv2
import numpy as np
from ultralytics import YOLO

# CONFIGURATION
VIDEO_PATH = "entrance_video.mp4"

MODEL_PATH = "yolov8n.pt"

CONFIDENCE = 0.40

# Heatmap intensity
DECAY = 0.97

# LOAD YOLO MODEL
model = YOLO(MODEL_PATH)
print("YOLO model loaded successfully!")

# OPEN VIDEO
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise ValueError("Unable to open video!")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Video loaded successfully!")

# Initialize heatmap
heatmap = np.zeros((height, width), dtype=np.float32)

# MAIN PROCESSING LOOP
while True:

    ret, frame = cap.read()

    if not ret:
        print("Video completed!")
        break

    # YOLO HUMAN DETECTION
    results = model.predict(
        source=frame,
        conf=CONFIDENCE,
        classes=[0],
        verbose=False
    )

    human_count = 0

    # Reduce previous heat gradually
    heatmap *= DECAY

    for result in results:

        if result.boxes is None:
            continue

        for box in result.boxes:

            x1, y1, x2, y2 = map(
                int,
                box.xyxy[0].tolist()
            )

            confidence = float(box.conf[0])

            human_count += 1

            # Calculate center point
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2

            # Add activity to heatmap
            cv2.circle(
                heatmap,
                (center_x, center_y),
                45,
                1.0,
                -1
            )

            # Draw bounding box
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            # Display confidence
            cv2.putText(
                frame,
                f"Human {confidence:.2f}",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # 6. GENERATE HEATMAP
    smooth_heatmap = cv2.GaussianBlur(
        heatmap,
        (51, 51),
        0
    )

    normalized = cv2.normalize(
        smooth_heatmap,
        None,
        0,
        255,
        cv2.NORM_MINMAX
    )

    normalized = normalized.astype(np.uint8)

    colored_heatmap = cv2.applyColorMap(
        normalized,
        cv2.COLORMAP_JET
    )

    # COMBINE VIDEO AND HEATMAP
    output = cv2.addWeighted(
        frame,
        0.65,
        colored_heatmap,
        0.35,
        0
    )

    # CROWD DENSITY ANALYSIS
    if human_count <= 2:
        density = "LOW"

        color = (0, 255, 0)

    elif human_count <= 5:
        density = "MEDIUM"

        color = (0, 255, 255)

    else:
        density = "HIGH"

        color = (0, 0, 255)

    # DISPLAY SECURITY DASHBOARD
    cv2.rectangle(
        output,
        (10, 10),
        (390, 125),
        (30, 30, 30),
        -1
    )

    cv2.putText(
        output,
        "HUMAN ACTIVITY HEATMAP",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        output,
        f"Humans Detected: {human_count}",
        (20, 75),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )

    cv2.putText(
        output,
        f"Crowd Density: {density}",
        (20, 105),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        color,
        2
    )

    # DISPLAY OUTPUT
    cv2.imshow(
        "Activity Heatmap",
        output
    )

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# RELEASE RESOURCES
cap.release()

cv2.destroyAllWindows()


YOLO model loaded successfully!
Video loaded successfully!


KeyboardInterrupt: 